# AquaPINN FishHeart Full Pipeline

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwq2016/AquaPINN_tmp/blob/main/Examples/Colab/03_FishHeart_Full_Pipeline.ipynb)

Run the cells from top to bottom. This notebook reads `Examples/fishHeart/run/params.toml` directly; it has no separate parameter form. Edit that TOML file before running if you need to change inputs, tag, time window, method, or output folder.


## 1. Set up

Run the setup cell once. It installs dependencies, finds or clones the project, and mounts Google Drive in Colab. Each completed run is copied into a separate Full Study folder so earlier results are preserved.


In [ ]:
%pip -q install "numpy<2.5" scipy pandas matplotlib utm networkx scikit-learn tqdm six

from pathlib import Path
import subprocess, sys

# Use this checkout, or clone the public repository when running in Colab.
REPO_URL = "https://github.com/cwq2016/AquaPINN_tmp.git"
REPO_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "AquaPINN_Tracker3D").is_dir()), None)
if REPO_DIR is None:
    REPO_DIR = Path("/content/AquaPINN_tmp")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                        REPO_URL, str(REPO_DIR)], check=True)

# Colab runtime files are temporary; save completed results to Drive.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT = Path("/content/drive/MyDrive/AquaPINN_Results")
except ImportError:
    DRIVE_OUTPUT = REPO_DIR / "AquaPINN_Colab_Runs"
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
print("Project ready:", REPO_DIR)
print("Saved run bundles:", DRIVE_OUTPUT)


## 2. Load settings and run the pipeline

The notebook runs from `Examples/fishHeart/run` so the relative paths in `params.toml` resolve as intended. It then calls the existing source functions in order: deploy the receiver system, decode recordings, synchronize clocks, and track the selected tag. You do not need to edit this cell.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, sys, uuid
import pandas as pd
import tomllib

# The launchers expect relative paths to be resolved from their run folder.
FISH_RUN = REPO_DIR / "Examples/fishHeart/run"
sys.path.insert(0, str(REPO_DIR))
os.chdir(FISH_RUN)

import AquaPINN_Tracker3D as pinn
from AquaPINN_Tracker3D import config
from AquaPINN_Tracker3D.SharedLibs.functions import load_params

PARAMS_FILE = FISH_RUN / "params.toml"
# Read both the original TOML values and the first concrete parameter combination.
with PARAMS_FILE.open("rb") as stream:
    study_parameters = tomllib.load(stream)
train_params = load_params(str(PARAMS_FILE))[0]
tag_id = int(train_params["TagID"])
output_dir = Path(train_params["output_folder"])
if not output_dir.is_absolute():
    output_dir = (FISH_RUN / output_dir).resolve()
output_dir.mkdir(parents=True, exist_ok=True)
print("Loaded:", PARAMS_FILE)
print("TagID:", tag_id, "| Method:", train_params["trackMethod"],
      "| Output folder:", output_dir)

# Run the same three source functions as the FishHeart launcher.
# Initialize AquaPINN and write its history log into the configured output folder.
config.oneKey_configure(use_cuda=True, precision=64, seed=42)
pinn.getLogger(logFile=str(output_dir / f"history_Tag{tag_id}.log"))
# FishHeart starts from raw receiver files, so clock synchronization is enabled.
loc_sys = pinn.localizationSystem().deploy(str(PARAMS_FILE), doSync=True)
# Stage 1: decode raw SR*.csv receiver recordings into tag detections.
pinn.decoder.run(str(PARAMS_FILE), loc_sys)
# Stage 2: estimate and apply receiver clock offsets from beacon detections.
pinn.syncer.run(str(PARAMS_FILE), loc_sys)
# Stage 3: estimate the selected tag trajectory using the configured method.
pinn.tracker.run(str(PARAMS_FILE), loc_sys)

# Bundle the configured output and its desktop Full Study metadata.
tag_code = str(loc_sys.tagInfo["code"][tag_id]).strip()
method = "NNplus" if train_params["trackMethod"] == "NN" and train_params.get("nSamples", 1) > 1 else train_params["trackMethod"]
track_name = f"track_Tag{tag_id}_{tag_code}_{method}.csv"
TRACK_CSV = output_dir / "Track" / "track" / track_name
run_id = uuid.uuid4().hex
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
SAVED_RUN = DRIVE_OUTPUT / f"FishHeart_{stamp}_{run_id[:8]}"
# Start a unique result bundle, then place engine outputs under processing/.
SAVED_RUN.mkdir(parents=True, exist_ok=False)
processing = SAVED_RUN / "processing"
shutil.copytree(output_dir, processing)
shutil.copy2(PARAMS_FILE, SAVED_RUN / "params.toml")

track_copy = processing / "Track" / "track" / track_name
track_table = pd.read_csv(track_copy)
track_summary = {
    "csv_sha256": hashlib.sha256(track_copy.read_bytes()).hexdigest(),
    "rows": int(len(track_table)),
    "columns": [str(column) for column in track_table.columns],
    "first_datetime": str(track_table["datetime"].dropna().iloc[0]) if "datetime" in track_table and track_table["datetime"].notna().any() else None,
    "last_datetime": str(track_table["datetime"].dropna().iloc[-1]) if "datetime" in track_table and track_table["datetime"].notna().any() else None,
}
# The Full Study record lists every tag and marks the configured TagID as selected.
tag_table = pd.read_csv(Path(train_params["tagFile"]))
study_tags = []
for row, item in tag_table.iterrows():
    study_tags.append({
        "code": str(item["Code"]).strip(), "row": int(row),
        "nominal_pri": float(item["PRI"]),
        "role": "fish" if int(row) == tag_id else "unassigned",
        "selected": int(row) == tag_id, "gps": None,
    })
samples = int(train_params.get("nSamples", 1))
# These fields let the desktop Full Study viewer reopen this notebook's output.
study = {
    "format": "aquapinn-study", "version": 1, "name": "FishHeart",
    "workflow": "fish", "method": method, "nnplus_samples": max(2, samples),
    "parameters": study_parameters, "tags": study_tags,
    "unknown_parameters": {}, "source_config": "params.toml",
    "import_notes": ["Loaded directly from the FishHeart run params.toml."],
    "coordinate_frame": "FishHeart receiver coordinates as supplied; confirm projection and datum for scientific interpretation.",
    "sync_support_reviewed": True,
    "gps_draft": {"file": str(train_params.get("GPSFile", "")),
                   "utc_offset": int(train_params["timeZone"]),
                   "matching_frame": False, "confirmed": False,
                   "start": "", "end": "", "target_codes": []},
    "evaluation": {"window_seconds": 90, "step_seconds": 1,
                   "max_gap_seconds": 5, "constant_pri_confirmed": False},
    "example": False,
}
track_rel = f"processing/Track/track/{track_name}"
job = {**study_tags[tag_id], "method": method, "state": "completed",
       "track_csv": track_rel, "track_sha256": track_summary["csv_sha256"],
       "summary": track_summary}
# The job entry links the selected tag to its saved track and summary.
run_record = {"format": "aquapinn-study-run", "version": 1,
              "run_id": run_id, "state": "completed", "method": method,
              "study": study, "jobs": [job]}
(SAVED_RUN / "study.json").write_text(json.dumps(study, indent=2), encoding="utf-8")
(SAVED_RUN / "run.json").write_text(json.dumps(run_record, indent=2), encoding="utf-8")
print("Full Study run saved:", SAVED_RUN)


## 3. Review results

The saved Full Study folder contains `processing/` with decoded data, synchronization products, tracks, figures, and the history log. `study.json` and `run.json` describe the run for the desktop viewer; the copied `params.toml` records the settings used.


In [ ]:
import pandas as pd
from IPython.display import display, Image

# Preview the selected tag's track and figures from the configured output folder.
track = pd.read_csv(TRACK_CSV)
display(track.head())
for figure in sorted(output_dir.rglob("*.jpg"))[:4]:
    display(Image(filename=str(figure)))
print("Saved Full Study folder:", SAVED_RUN)
print("Run record:", SAVED_RUN / "run.json")
